# Start From Here

Starter notebook for the UW–IBM Quantum Machine Learning Hackathon. Read the companion notebook `UW-IBM_QML_Hackathon.ipynb`to understand the hackathon objectives. This notebook handles environment setup, data preprocessing, and a quick baseline so you can hit the ground running:

1. Install + imports (Sections 1, 2)
2. Download + preprocess the CAR T-cell data (Section 3)
3. Quick baseline — classical vs. quantum-projected SVM with F1 comparison and kernel heatmaps (Section 4)
4. Pointer to the IBM PQK tutorial for Tracks 1, 2, and Bonus

After running every cell here, you'll have the same variables the IBM PQK tutorial expects from Step 2 onwards, plus a ΔF1 result you can sanity-check against the tutorial. From there you copy cells from the tutorial to build out your own analyses for the hackathon prompt's challenge tracks.

## 1. Environment Setup

### Install required packages

Run this once per environment. If you're on Colab or a fresh JupyterLab session, this will take ~2 minutes. If everything is already installed, pip will skip ahead.

In [ ]:
!pip install -U "qiskit[visualization]" qiskit-ibm-runtime \
    category-encoders==2.8.1 numpy==2.3.2 pandas==2.3.2 \
    scikit-learn tqdm==4.67.1

### Import libraries

All imports for the whole notebook live in this single cell. If you run a later cell and get a `NameError`, come back and re-run this one.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine learning
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

from scipy.linalg import inv, sqrtm

# Qiskit (only needed for Track 2D real-hardware comparison)
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import (
    Batch,
    EstimatorOptions,
    EstimatorV2 as Estimator,
    QiskitRuntimeService,
)

import tqdm

# Global random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. IBM Quantum Account Setup

To run the QPU hardware sections, you need access to an IBM Quantum account or team instance. **Do not paste or commit API keys in this notebook.**

If you need to authenticate for the first time, run the following once in a private local setup notebook or Python shell, not in the public GitHub notebook:

```python
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="YOUR_IBM_QUANTUM_API_KEY",
    overwrite=True
)
```

After your credentials are saved locally, this notebook can connect without exposing your token.

> Note: If you are using a hackathon/team instance, make sure your local IBM Quantum account is configured for that instance. Team accounts may have QPU time limits or expiration dates, so save results locally after hardware jobs finish.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Loads credentials saved locally on your machine.
# This cell should not contain API keys, tokens, or CRNs.
service = QiskitRuntimeService()

print("Available backends:", [b.name for b in service.backends()])


## 3. Download and preprocess the CAR T-cell data

The dataset is hosted in the official Qiskit documentation repo. We download four files:

- `train_data.csv`, `test_data.csv` — raw motif IDs and cytotoxicity scores
- `projections_train.csv`, `projections_test.csv` — pre-computed 180-dim quantum projections from IBM Heron R2 hardware

After downloading, we sanitize the files if needed (some shipped versions have a UTF-8 BOM and wrapped quotes), then preprocess them using the **exact pipeline from the IBM PQK tutorial** — same `preprocess_data` / `data_encoder` functions, same `args` dict, same π/2 scaling — so your numbers align with the paper (Utro et al. 2025) and the official tutorial.

In [ ]:
def download_pqk_dataset(data_dir="data_tutorial/pqk"):
    """Download the four CSV files from the Qiskit documentation repo."""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    base_url = (
        "https://raw.githubusercontent.com/Qiskit/documentation/main/"
        "datasets/tutorials/pqk"
    )
    files = [
        "train_data.csv",
        "test_data.csv",
        "projections_train.csv",
        "projections_test.csv",
    ]

    for filename in files:
        url = f"{base_url}/{filename}"
        dest = data_dir / filename
        print(f"  {filename} ...", end=" ", flush=True)
        urllib.request.urlretrieve(url, dest)
        print(f"✓ ({dest.stat().st_size:,} bytes)")

    print(f"\nAll files saved to {data_dir}/")
    return data_dir


DATA_DIR = download_pqk_dataset()

In [ ]:
# ---------------------------------------------------------------------------
# Helper: sanitize CSV files into well-formed ones.
#
# The data files in this dataset sometimes ship with two quirks that confuse pd.read_csv:
#   - a UTF-8 BOM at the start
#   - every line wrapped in an extra pair of double quotes, so the whole row is parsed as one column of 172/74 strings (motif CSVs only)
#   - whitespace-separated floats instead of commas (projection CSVs only)
#
# We detect and repair these in place so the IBM tutorial code below works unmodified. Already well-formed files are left alone.
# ---------------------------------------------------------------------------
def sanitize_csv_files(data_dir):
    """Normalize quirky CSV files to well-formed UTF-8 comma-separated CSVs."""
    data_dir = Path(data_dir)

    # Motif files: strip BOM and outer line-wrap quotes if present.
    for name in ["train_data.csv", "test_data.csv"]:
        path = data_dir / name
        if not path.exists():
            continue
        with open(path, "rb") as f:
            raw = f.read()
        text = raw.decode("utf-8-sig")
        lines = text.replace("\r\n", "\n").strip().split("\n")
        wrapped = all(
            len(L) >= 2 and L.startswith('"') and L.endswith('"') for L in lines
        )
        if wrapped:
            lines = [L[1:-1] for L in lines]
        cleaned = "\n".join(lines) + "\n"
        if cleaned.encode("utf-8") != raw:
            path.write_text(cleaned, encoding="utf-8")
            print(f"  Sanitized {name}")

    # Projection files: convert whitespace-separated to comma-separated if needed.
    for name in ["projections_train.csv", "projections_test.csv"]:
        path = data_dir / name
        if not path.exists():
            continue
        with open(path, "rb") as f:
            head = f.read(2048)
        text_head = head.decode("utf-8-sig", errors="replace")
        first_line = text_head.split("\n")[0]
        # If the first line has many whitespace-separated floats and no commas,
        # it\'s the whitespace-separated format.
        if "," not in first_line and len(first_line.split()) > 10:
            # Whitespace-separated → reload with pandas sep=r"\s+" and rewrite as CSV.
            from io import StringIO
            with open(path, "rb") as f:
                text = f.read().decode("utf-8-sig")
            df = pd.read_csv(StringIO(text), sep=r"\s+", header=None, engine="python")
            df.to_csv(path, index=False, header=False)
            print(f"  Sanitized {name}")


sanitize_csv_files(DATA_DIR)


# ---------------------------------------------------------------------------
# The two preprocessing functions below are the IBM PQK tutorial code, 
# kept intentionally close to the original so your results align with the paper (Utro et al. 2025) and the tutorial. See:
#   https://quantum.cloud.ibm.com/docs/en/tutorials/projected-quantum-kernels
# ---------------------------------------------------------------------------
def preprocess_data(dir_root, args):
    """
    Preprocess the training and test data.
    Tutorial-exact: load CSVs, normalize motif ID 17 → 14, force canonical
    column names, optionally filter by spacer-motif rules, binarize labels
    at the cytotoxicity threshold, shift motif IDs to start at 0.
    """
    # Read from the csv files
    train_data = pd.read_csv(
        os.path.join(dir_root, args["file_train_data"]),
        encoding="unicode_escape",
        sep=",",
    )
    test_data = pd.read_csv(
        os.path.join(dir_root, args["file_test_data"]),
        encoding="unicode_escape",
        sep=",",
    )

    # Fix the last motif ID
    train_data[train_data == 17] = 14
    train_data.columns = [
        "Cell Number",
        "motif",
        "motif.1",
        "motif.2",
        "motif.3",
        "motif.4",
        "Nalm 6 Cytotoxicity",
    ]
    test_data[test_data == 17] = 14
    test_data.columns = [
        "Cell Number",
        "motif",
        "motif.1",
        "motif.2",
        "motif.3",
        "motif.4",
        "Nalm 6 Cytotoxicity",
    ]

    # Adjust motif at the third position
    if args["filter_for_spacer_motif_third_position"]:
        train_data = train_data[
            (train_data["motif.2"] == 14) | (train_data["motif.2"] == 0)
        ]
        test_data = test_data[
            (test_data["motif.2"] == 14) | (test_data["motif.2"] == 0)
        ]

    train_data = train_data[
        args["motifs_to_use"] + [args["label_name"], "Cell Number"]
    ]
    test_data = test_data[
        args["motifs_to_use"] + [args["label_name"], "Cell Number"]
    ]

    # Adjust motif at the last position
    if not args["allow_spacer_motif_last_position"]:
        last_motif = args["motifs_to_use"][len(args["motifs_to_use"]) - 1]
        train_data = train_data[
            (train_data[last_motif] != 14) & (train_data[last_motif] != 0)
        ]
        test_data = test_data[
            (test_data[last_motif] != 14) & (test_data[last_motif] != 0)
        ]

    # Get the labels
    train_labels = np.array(train_data[args["label_name"]])
    test_labels = np.array(test_data[args["label_name"]])

    # For the classification task use the threshold to binarize labels
    train_labels[train_labels > args["label_binarization_threshold"]] = 1
    train_labels[train_labels < 1] = args["min_label_value"]
    test_labels[test_labels > args["label_binarization_threshold"]] = 1
    test_labels[test_labels < 1] = args["min_label_value"]

    # Reduce data to just the motifs of interest
    train_data = train_data[args["motifs_to_use"]]
    test_data = test_data[args["motifs_to_use"]]

    # Get the class and motif counts
    min_class = np.min(np.unique(np.concatenate([train_data, test_data])))
    max_class = np.max(np.unique(np.concatenate([train_data, test_data])))

    num_class = max_class - min_class + 1
    num_motifs = len(args["motifs_to_use"])
    print(f"  max_class:min_class:num_class = {max_class}:{min_class}:{num_class}")
    print(f"  Train: {len(train_data)} samples, Test: {len(test_data)} samples")

    train_data = train_data - min_class
    test_data = test_data - min_class

    return (
        train_data,
        test_data,
        train_labels,
        test_labels,
        num_class,
        num_motifs,
    )


def data_encoder(args, train_data, test_data, num_class, num_motifs):
    """
    Use one-hot or binary encoding for classical data representation.
    Tutorial-exact.
    """
    if args["encoder"] == "one-hot":
        train_data = np.eye(num_class)[train_data]
        test_data = np.eye(num_class)[test_data]

        train_data = train_data.reshape(
            train_data.shape[0], train_data.shape[1] * train_data.shape[2]
        )
        test_data = test_data.reshape(
            test_data.shape[0], test_data.shape[1] * test_data.shape[2]
        )

    elif args["encoder"] == "binary":
        import category_encoders as ce
        encoder = ce.BinaryEncoder()
        base_array = np.unique(np.concatenate([train_data, test_data]))
        base = pd.DataFrame(base_array).astype("category")
        base.columns = ["motif"]
        for motif_name in args["motifs_to_use"][1:]:
            base[motif_name] = base.loc[:, "motif"]
        encoder.fit(base)

        train_data = encoder.transform(train_data.astype("category"))
        test_data = encoder.transform(test_data.astype("category"))

        train_data = np.reshape(
            train_data.values, (train_data.shape[0], num_motifs, -1)
        )
        test_data = np.reshape(
            test_data.values, (test_data.shape[0], num_motifs, -1)
        )

        train_data = train_data.reshape(
            train_data.shape[0], train_data.shape[1] * train_data.shape[2]
        )
        test_data = test_data.reshape(
            test_data.shape[0], test_data.shape[1] * test_data.shape[2]
        )

    else:
        raise ValueError("Invalid encoding type.")

    return train_data, test_data


# ---------------------------------------------------------------------------
# Run preprocessing using the tutorial's exact args.
# ---------------------------------------------------------------------------
args = {
    "file_train_data": "train_data.csv",
    "file_test_data": "test_data.csv",
    "motifs_to_use": ["motif", "motif.1", "motif.2", "motif.3"],
    "label_name": "Nalm 6 Cytotoxicity",
    "label_binarization_threshold": 0.62,
    "filter_for_spacer_motif_third_position": False,
    "allow_spacer_motif_last_position": True,
    "min_label_value": -1,
    "encoder": "one-hot",
}

# preprocess_data returns motif-ID DataFrames; we keep these around for Track 2C.
train_motifs, test_motifs, train_labels, test_labels, num_class, num_motifs = (
    preprocess_data(dir_root=str(DATA_DIR), args=args)
)
NUM_CLASS = int(num_class)
NUM_MOTIFS = int(num_motifs)

# data_encoder turns motif IDs into one-hot vectors.
train_data, test_data = data_encoder(args, train_motifs, test_motifs, num_class, num_motifs)

# Tutorial step: scale the active one-hot entries to π/2.
angle = np.pi / 2
tmp = pd.DataFrame(train_data).astype("float64"); tmp[tmp == 1] = angle
train_data = tmp.values
tmp = pd.DataFrame(test_data).astype("float64"); tmp[tmp == 1] = angle
test_data = tmp.values

print(f"  train_data: {train_data.shape}")
print(f"  test_data:  {test_data.shape}")

### Load the pre-computed quantum projections

The IBM PQK tutorial spends roughly 80 minutes of QPU time generating the 180-dim projections. We use the pre-computed CSVs that shipped with the tutorial repo, so you can skip directly to analysis.

After this cell runs, you'll have all the variables the IBM tutorial expects from Step 2 onwards:

| Variable | Shape | Source |
|---|---|---|
| `train_data` | `(172, 60)` | One-hot motifs × π/2 |
| `test_data` | `(74, 60)` | Same encoding |
| `train_labels` | `(172,)` | Binary {−1, +1} |
| `test_labels` | `(74,)` | Same |
| `projections_train` | `(172, 180)` | Pre-computed quantum 1-RDMs |
| `projections_test` | `(74, 180)` | Same |
| `num_class` | `15` | Number of motif IDs |
| `num_motifs` | `4` | Number of motif positions |

If you want to generate your own projections on a real QPU as part of the hackathon, follow the tutorial's Step 3 ("Execute using Qiskit primitives"). You'll need an active IBM Quantum account and ~80 minutes of queue + execution time on a Heron-class backend.

In [ ]:
# Load the pre-computed 180-dim quantum projections.
# Headers may or may not be present depending on which version of the file
# you have; we read with header=0 first and detect the no-header case by
# checking that the first row is numeric.
def _load_projections(path):
    df = pd.read_csv(path)
    # If the header row turned out to be data (first column not parseable as
    # column-name-like), reload without a header.
    try:
        _ = df.columns.astype(float)
        df = pd.read_csv(path, header=None)
    except (ValueError, TypeError):
        pass
    df.columns = [f"q{i}" for i in range(df.shape[1])]
    return df


projections_train = _load_projections(DATA_DIR / "projections_train.csv")
projections_test = _load_projections(DATA_DIR / "projections_test.csv")

# Sanity check — all four feature matrices and the label arrays must agree.
assert len(train_data) == len(projections_train) == len(train_labels), (
    f"Train length mismatch: train_data={len(train_data)}, "
    f"projections_train={len(projections_train)}, labels={len(train_labels)}"
)
assert len(test_data) == len(projections_test) == len(test_labels), (
    f"Test length mismatch: test_data={len(test_data)}, "
    f"projections_test={len(projections_test)}, labels={len(test_labels)}"
)

print(f"train_data:        {train_data.shape},   test_data:        {test_data.shape}")
print(f"projections_train: {projections_train.shape}, projections_test: {projections_test.shape}")
print(f"train_labels:      {train_labels.shape},        test_labels:      {test_labels.shape}")
print(f"Class balance (train): {dict(zip(*np.unique(train_labels, return_counts=True)))}")
print(f"Class balance (test):  {dict(zip(*np.unique(test_labels, return_counts=True)))}")


## 4. Quick baseline

This section trains a classical and a quantum-projected RBF-SVM on the data you just loaded and compares them. It uses the same hyperparameter grid as the IBM PQK tutorial (13 × 12 = 156 grid points, 10-fold CV) so your numbers will match the tutorial's headline result.

Runtime: ~1–2 minutes on a modern laptop.

After this you have the **Track 0 baseline result** plus two visuals to look at, and you can move on to designing your own Track 0 / 1 / 2 / Bonus experiments by copying cells from the tutorial.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Hyperparameter grid matched to the IBM PQK tutorial scale.
# 13 C values × 12 γ values = 156 grid points, 10-fold CV.
PARAM_GRID = {
    "C":     [0.1, 0.5, 1, 2.5, 5, 7.5, 8.5, 10, 10.75, 15, 25, 50, 100],
    "gamma": ["auto", "scale", 0.001, 0.005, 0.01, 0.02, 0.04, 0.05,
              0.1, 0.25, 0.5, 1],
}
cv = StratifiedKFold(n_splits=10)

# Classical SVM on raw one-hot features
print("Training classical SVM (~30s)...")
svm_classical = GridSearchCV(
    SVC(kernel="rbf"), PARAM_GRID, cv=cv,
    scoring="f1_weighted", n_jobs=-1,
).fit(train_data, train_labels)
pred_classical = svm_classical.predict(test_data)
f1_classical = f1_score(test_labels, pred_classical, average="weighted")

# Quantum-projected SVM on the pre-computed 180-dim projections
print("Training quantum-projected SVM (~30s)...")
svm_quantum = GridSearchCV(
    SVC(kernel="rbf"), PARAM_GRID, cv=cv,
    scoring="f1_weighted", n_jobs=-1,
).fit(projections_train, train_labels)
pred_quantum = svm_quantum.predict(projections_test)
f1_quantum = f1_score(test_labels, pred_quantum, average="weighted")

print()
print(f"Classical: best params = {svm_classical.best_params_}, test F1 = {f1_classical:.4f}")
print(f"Quantum:   best params = {svm_quantum.best_params_}, test F1 = {f1_quantum:.4f}")
print(f"ΔF1 (quantum − classical) = {f1_quantum - f1_classical:+.4f}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Classical", "Quantum-projected"],
    [f1_classical, f1_quantum],
    color=["#3498db", "#e74c3c"], alpha=0.85, edgecolor="black",
)
for b, score in zip(bars, [f1_classical, f1_quantum]):
    ax.text(b.get_x() + b.get_width() / 2, score + 0.01,
            f"{score:.4f}", ha="center", fontweight="bold")
ax.set_ylabel("Weighted F1 (test)")
ax.set_ylim(0, 1)
ax.set_title(f"Track 0 baseline — ΔF1 = {f1_quantum - f1_classical:+.4f}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel


def resolve_gamma(gamma, X):
    """Convert sklearn's 'scale'/'auto' strings to a numeric γ for rbf_kernel."""
    Xarr = np.asarray(X)
    if gamma == "scale":
        return 1.0 / (Xarr.shape[1] * Xarr.var())
    if gamma == "auto":
        return 1.0 / Xarr.shape[1]
    return gamma


gamma_c = resolve_gamma(svm_classical.best_params_["gamma"], train_data)
gamma_q = resolve_gamma(svm_quantum.best_params_["gamma"], projections_train)

K_c = rbf_kernel(train_data, gamma=gamma_c)
K_q = rbf_kernel(projections_train, gamma=gamma_q)

# Sort rows and columns by label to make any class structure visible
sort_idx = np.argsort(train_labels)
K_c_sorted = K_c[sort_idx][:, sort_idx]
K_q_sorted = K_q[sort_idx][:, sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, K, title in zip(
    axes,
    [K_c_sorted, K_q_sorted],
    ["Classical kernel (sorted by label)",
     "Quantum-projected kernel (sorted by label)"],
):
    im = ax.imshow(K, cmap="viridis", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Training sample")
    ax.set_ylabel("Training sample")
    plt.colorbar(im, ax=ax, label="Kernel value")
plt.tight_layout()
plt.show()

print("Look for block-diagonal structure (high within-class similarity)."
      " Which kernel shows it more clearly?")


## You're ready — continue from the IBM PQK tutorial

You now have a working baseline (the section above). To go further on Tracks 1 (Anatomy), 2 (Robustness), or Bonus (geometric difference), open the IBM PQK tutorial notebook (`projected-quantum-kernels.ipynb`) and copy cells into this notebook (or a scratch notebook in the same kernel session).

### Which tutorial cells to copy, and which to SKIP

The tutorial mixes "explain how to compute the projections" with "use the pre-computed projections". You already have the projections — you should **skip the cells that re-compute them**. Here's a precise guide:

| Tutorial cell | Content | What to do |
|---|---|---|
| **Cell 20** | `feature_dimension = train_data.shape[1]` + ZZFeatureMap embedding | Copy if you want to use the ZZ feature map |
| **Cell 22** | Same as 20 but with 1D-Heisenberg Hamiltonian embedding | Copy (optional, alternative embedding) |
| **Cells 27, 29, 31, 32, 34** | Backend selection, single-sample QPU smoke test | Skip unless you want to verify QPU access |
| **Cells 37–44** | Loop over all 172 training samples on QPU (~80 min) | Skip — these regenerate the projections we already loaded |
| **Cell 51** | `projections_train = np.loadtxt("projections_train.txt")` | SKIP — will fail. This loads cached `.txt` files that don't ship with the dataset; we already loaded the `.csv` versions above into `projections_train` and `projections_test`. |
| **Cells 54, 56** | Train classical + quantum SVMs with GridSearchCV | ✅ Copy — this is your Track 0 baseline |
| **Cells 59, 61, 62** | Geometric difference $g$ + model complexities | ✅ Copy — this is the Bonus track |

### For Tracks 1 and 2 (Anatomy / Robustness)

Design your own slicing and subsetting code on top of `projections_train`, `projections_test`, `train_data`, `test_data`, and the motif metadata in `train_motifs`, `test_motifs`. Useful slicing patterns:

```python
# By measurement basis — projections_train is laid out as <X> | <Y> | <Z>
basis_X = projections_train.iloc[:, 0:60]      # ⟨X⟩ on 60 qubits
basis_Y = projections_train.iloc[:, 60:120]    # ⟨Y⟩
basis_Z = projections_train.iloc[:, 120:180]   # ⟨Z⟩

# By motif position — every 15-column block within each basis
# (position 1 = cols 0-14, position 2 = cols 15-29, etc., per basis)
```

For the problem statement, hypotheses, and presentation requirements, see the companion `UW-IBM_QML_Hackathon.ipynb` prompt notebook.

In [ ]:
print("train_data:", train_data.shape)
print("test_data:", test_data.shape)
print("projections_train:", projections_train.shape)
print("projections_test:", projections_test.shape)
print("train_labels:", train_labels.shape)
print("test_labels:", test_labels.shape)

In [ ]:
# Track 1B: slice quantum features by CAR-T position
# Each position has 15 features inside each measurement block.
# Z block: columns 0-59
# X block: columns 60-119
# Y block: columns 120-179

position_slices = {
    "Position 1": list(range(0, 15)) + list(range(60, 75)) + list(range(120, 135)),
    "Position 2": list(range(15, 30)) + list(range(75, 90)) + list(range(135, 150)),
    "Position 3": list(range(30, 45)) + list(range(90, 105)) + list(range(150, 165)),
    "Position 4": list(range(45, 60)) + list(range(105, 120)) + list(range(165, 180)),
}

for position, cols in position_slices.items():
    print(position, len(cols), "features")

In [ ]:
# Extract only Position 1 quantum features

pos1_cols = position_slices["Position 1"]

X_train_pos1 = projections_train.iloc[:, pos1_cols]
X_test_pos1 = projections_test.iloc[:, pos1_cols]

print("X_train_pos1:", X_train_pos1.shape)
print("X_test_pos1:", X_test_pos1.shape)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Same kind of RBF-SVM as baseline, but only using Position 1 quantum features
param_grid_pos = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

svm_pos1 = GridSearchCV(
    SVC(kernel="rbf"),
    param_grid_pos,
    cv=10,
    scoring="f1_weighted",
    n_jobs=-1
)

svm_pos1.fit(X_train_pos1, train_labels)

pred_pos1 = svm_pos1.predict(X_test_pos1)
f1_pos1 = f1_score(test_labels, pred_pos1, average="weighted")

print("Position 1 best params:", svm_pos1.best_params_)
print("Position 1 test F1:", round(f1_pos1, 4))

In [ ]:
# Extract only Position 2 quantum features

pos2_cols = position_slices["Position 2"]

X_train_pos2 = projections_train.iloc[:, pos2_cols]
X_test_pos2 = projections_test.iloc[:, pos2_cols]

print("X_train_pos2:", X_train_pos2.shape)
print("X_test_pos2:", X_test_pos2.shape)

In [ ]:
# Train SVM using only Position 2 quantum features

svm_pos2 = GridSearchCV(
    SVC(kernel="rbf"),
    param_grid_pos,
    cv=10,
    scoring="f1_weighted",
    n_jobs=-1
)

svm_pos2.fit(X_train_pos2, train_labels)

pred_pos2 = svm_pos2.predict(X_test_pos2)
f1_pos2 = f1_score(test_labels, pred_pos2, average="weighted")

print("Position 2 best params:", svm_pos2.best_params_)
print("Position 2 test F1:", round(f1_pos2, 4))

In [ ]:
# Extract only Position 3 quantum features

pos3_cols = position_slices["Position 3"]

X_train_pos3 = projections_train.iloc[:, pos3_cols]
X_test_pos3 = projections_test.iloc[:, pos3_cols]

print("X_train_pos3:", X_train_pos3.shape)
print("X_test_pos3:", X_test_pos3.shape)

In [ ]:
# Train SVM using only Position 3 quantum features

svm_pos3 = GridSearchCV(
    SVC(kernel="rbf"),
    param_grid_pos,
    cv=10,
    scoring="f1_weighted",
    n_jobs=-1
)

svm_pos3.fit(X_train_pos3, train_labels)

pred_pos3 = svm_pos3.predict(X_test_pos3)
f1_pos3 = f1_score(test_labels, pred_pos3, average="weighted")

print("Position 3 best params:", svm_pos3.best_params_)
print("Position 3 test F1:", round(f1_pos3, 4))

In [ ]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

position_results = []
position_models = {}

for position_name, cols in position_slices.items():
    print(f"Training {position_name}...")

    # Select only this position's 45 quantum features
    X_train_pos = projections_train.iloc[:, cols]
    X_test_pos = projections_test.iloc[:, cols]

    # Train SVM
    svm_pos = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid_pos,
        cv=10,
        scoring="f1_weighted",
        n_jobs=-1
    )

    svm_pos.fit(X_train_pos, train_labels)

    # Test performance
    pred_pos = svm_pos.predict(X_test_pos)
    f1_pos = f1_score(test_labels, pred_pos, average="weighted")

    # Save results
    position_results.append({
        "Position": position_name,
        "Features Used": len(cols),
        "Best C": svm_pos.best_params_["C"],
        "Best gamma": svm_pos.best_params_["gamma"],
        "Test F1": round(f1_pos, 4)
    })

    position_models[position_name] = svm_pos

# Make results table
position_results_df = pd.DataFrame(position_results)
position_results_df = position_results_df.sort_values("Test F1", ascending=False)

position_results_df

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# -----------------------------
# 1. Combine train + test data
# -----------------------------
# We combine the official train/test split into one full dataset of 246 samples,
# then repeatedly create new random train/test splits.

X_quantum_all = pd.concat([projections_train, projections_test], axis=0).reset_index(drop=True)
y_all = np.concatenate([train_labels, test_labels])

print("X_quantum_all:", X_quantum_all.shape)
print("y_all:", y_all.shape)

# -----------------------------
# 2. Define position feature sets
# -----------------------------
# Each position uses 45 features:
# 15 from Z block + 15 from X block + 15 from Y block

position_slices = {
    "Position 1": list(range(0, 15)) + list(range(60, 75)) + list(range(120, 135)),
    "Position 2": list(range(15, 30)) + list(range(75, 90)) + list(range(135, 150)),
    "Position 3": list(range(30, 45)) + list(range(90, 105)) + list(range(150, 165)),
    "Position 4": list(range(45, 60)) + list(range(105, 120)) + list(range(165, 180)),
}

# -----------------------------
# 3. SVM hyperparameter grid
# -----------------------------
# Same idea as before, but not too huge so resampling finishes in reasonable time.

param_grid_pos = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

# -----------------------------
# 4. Repeated random splits
# -----------------------------
# n_splits = number of random resamples.
# test_size = 30% test, 70% train.
# Stratified means each split preserves the HIGH/LOW class balance.

n_splits = 20

splitter = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=0.30,
    random_state=42
)

all_results = []

for split_id, (train_idx, test_idx) in enumerate(splitter.split(X_quantum_all, y_all), start=1):
    print(f"\nRandom split {split_id}/{n_splits}")

    y_train_split = y_all[train_idx]
    y_test_split = y_all[test_idx]

    for position_name, cols in position_slices.items():
        X_train_split = X_quantum_all.iloc[train_idx, cols]
        X_test_split = X_quantum_all.iloc[test_idx, cols]

        svm = GridSearchCV(
            SVC(kernel="rbf"),
            param_grid_pos,
            cv=5,
            scoring="f1_weighted",
            n_jobs=-1
        )

        svm.fit(X_train_split, y_train_split)

        preds = svm.predict(X_test_split)
        f1 = f1_score(y_test_split, preds, average="weighted")

        all_results.append({
            "Split": split_id,
            "Position": position_name,
            "F1": f1,
            "Best C": svm.best_params_["C"],
            "Best gamma": svm.best_params_["gamma"]
        })

# -----------------------------
# 5. Summarize mean and std
# -----------------------------

resample_results_df = pd.DataFrame(all_results)

summary_df = (
    resample_results_df
    .groupby("Position")
    .agg(
        Mean_F1=("F1", "mean"),
        Std_F1=("F1", "std"),
        Min_F1=("F1", "min"),
        Max_F1=("F1", "max")
    )
    .reset_index()
)

summary_df["Mean_F1"] = summary_df["Mean_F1"].round(4)
summary_df["Std_F1"] = summary_df["Std_F1"].round(4)
summary_df["Min_F1"] = summary_df["Min_F1"].round(4)
summary_df["Max_F1"] = summary_df["Max_F1"].round(4)

summary_df = summary_df.sort_values("Mean_F1", ascending=False)

summary_df

In [ ]:
import matplotlib.pyplot as plt

plot_summary = summary_df.copy()
plot_summary["Position Number"] = plot_summary["Position"].str.extract(r"(\d+)").astype(int)
plot_summary = plot_summary.sort_values("Position Number")

plt.figure(figsize=(7, 4))

plt.bar(
    plot_summary["Position"],
    plot_summary["Mean_F1"],
    yerr=plot_summary["Std_F1"],
    capsize=5
)

plt.xlabel("CAR-T Motif Position")
plt.ylabel("Weighted F1 Score")
plt.title("Track 1B: Position Signal Across Random Resamples")
plt.ylim(0.40, 0.85)

plt.show()

In [ ]:
# Count which position wins on each random split

winner_df = (
    resample_results_df
    .loc[resample_results_df.groupby("Split")["F1"].idxmax()]
    [["Split", "Position", "F1"]]
)

win_counts = winner_df["Position"].value_counts().reset_index()
win_counts.columns = ["Position", "Number of Splits Won"]

win_counts

In [ ]:
import matplotlib.pyplot as plt

# Make sure all positions appear, even if they won 0 times
all_positions = pd.DataFrame({
    "Position": ["Position 1", "Position 2", "Position 3", "Position 4"]
})

win_counts_full = all_positions.merge(
    win_counts,
    on="Position",
    how="left"
).fillna(0)

win_counts_full["Number of Splits Won"] = win_counts_full["Number of Splits Won"].astype(int)

plt.figure(figsize=(7, 4))

plt.bar(
    win_counts_full["Position"],
    win_counts_full["Number of Splits Won"]
)

plt.xlabel("CAR-T Motif Position")
plt.ylabel("Number of Random Splits Won")
plt.title("Track 1B: Which Position Wins Across Random Resamples?")
plt.ylim(0, 20)

plt.show()

In [ ]:
## Track 1B Result: Position Ablation

We tested which CAR-T motif position carries the most predictive signal in the quantum-projected feature space. Each position used 45 quantum features: 15 motif features from each of the X, Y, and Z measurement blocks.

Across 20 stratified random train/test resamples, Position 2 had the strongest performance:

- Position 2: mean F1 = 0.7487 ± 0.0428
- Position 3: mean F1 = 0.7025 ± 0.0440
- Position 4: mean F1 = 0.6071 ± 0.0484
- Position 1: mean F1 = 0.5009 ± 0.0427

Position 2 also won 18 out of 20 random splits, while Position 3 won 2 out of 20. Positions 1 and 4 never won.

This suggests that the quantum-projected features associated with CAR-T Position 2 carry the strongest and most stable predictive signal.

In [ ]:
import numpy as np
import pandas as pd

def shuffle_position_blocks(X, random_state=None):
    """
    Shuffle the 4 CAR-T position blocks within each sample.

    X has 180 columns:
    - 3 measurement blocks of 60 columns each
    - inside each 60-column block:
        Position 1 = cols 0-14
        Position 2 = cols 15-29
        Position 3 = cols 30-44
        Position 4 = cols 45-59

    For each row/sample, we randomly permute the 4 position blocks.
    The same position permutation is applied across all 3 measurement blocks.
    """

    rng = np.random.default_rng(random_state)

    X_df = X.copy().reset_index(drop=True)
    X_shuffled = X_df.copy()

    measurement_block_starts = [0, 60, 120]
    position_size = 15
    num_positions = 4

    for row_idx in range(X_df.shape[0]):
        perm = rng.permutation(num_positions)

        for block_start in measurement_block_starts:
            original_blocks = []

            for pos in range(num_positions):
                start = block_start + pos * position_size
                end = start + position_size
                original_blocks.append(X_df.iloc[row_idx, start:end].to_numpy())

            for new_pos, old_pos in enumerate(perm):
                start = block_start + new_pos * position_size
                end = start + position_size
                X_shuffled.iloc[row_idx, start:end] = original_blocks[old_pos]

    return X_shuffled

In [ ]:
X_quantum_shuffled_test = shuffle_position_blocks(X_quantum_all, random_state=42)

print("Original shape:", X_quantum_all.shape)
print("Shuffled shape:", X_quantum_shuffled_test.shape)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

# We will compare original vs shuffled features across repeated random splits.

n_splits_2c = 20

splitter_2c = StratifiedShuffleSplit(
    n_splits=n_splits_2c,
    test_size=0.30,
    random_state=123
)

track2c_results = []

for split_id, (train_idx, test_idx) in enumerate(splitter_2c.split(X_quantum_all, y_all), start=1):
    print(f"\nTrack 2C split {split_id}/{n_splits_2c}")

    y_train_split = y_all[train_idx]
    y_test_split = y_all[test_idx]

    # -----------------------------
    # Condition 1: original quantum features
    # -----------------------------
    X_train_original = X_quantum_all.iloc[train_idx, :]
    X_test_original = X_quantum_all.iloc[test_idx, :]

    svm_original = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid_pos,
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1
    )

    svm_original.fit(X_train_original, y_train_split)

    pred_original = svm_original.predict(X_test_original)
    f1_original = f1_score(y_test_split, pred_original, average="weighted")

    track2c_results.append({
        "Split": split_id,
        "Condition": "Original",
        "F1": f1_original,
        "Best C": svm_original.best_params_["C"],
        "Best gamma": svm_original.best_params_["gamma"]
    })

    # -----------------------------
    # Condition 2: shuffled position blocks
    # -----------------------------
    X_quantum_shuffled = shuffle_position_blocks(
        X_quantum_all,
        random_state=1000 + split_id
    )

    X_train_shuffled = X_quantum_shuffled.iloc[train_idx, :]
    X_test_shuffled = X_quantum_shuffled.iloc[test_idx, :]

    svm_shuffled = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid_pos,
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1
    )

    svm_shuffled.fit(X_train_shuffled, y_train_split)

    pred_shuffled = svm_shuffled.predict(X_test_shuffled)
    f1_shuffled = f1_score(y_test_split, pred_shuffled, average="weighted")

    track2c_results.append({
        "Split": split_id,
        "Condition": "Shuffled positions",
        "F1": f1_shuffled,
        "Best C": svm_shuffled.best_params_["C"],
        "Best gamma": svm_shuffled.best_params_["gamma"]
    })

track2c_results_df = pd.DataFrame(track2c_results)

track2c_summary_df = (
    track2c_results_df
    .groupby("Condition")
    .agg(
        Mean_F1=("F1", "mean"),
        Std_F1=("F1", "std"),
        Min_F1=("F1", "min"),
        Max_F1=("F1", "max")
    )
    .reset_index()
)

track2c_summary_df["Mean_F1"] = track2c_summary_df["Mean_F1"].round(4)
track2c_summary_df["Std_F1"] = track2c_summary_df["Std_F1"].round(4)
track2c_summary_df["Min_F1"] = track2c_summary_df["Min_F1"].round(4)
track2c_summary_df["Max_F1"] = track2c_summary_df["Max_F1"].round(4)

track2c_summary_df

In [ ]:
# Compare Original vs Shuffled on each split

paired_2c = track2c_results_df.pivot(
    index="Split",
    columns="Condition",
    values="F1"
).reset_index()

paired_2c["Drop_Shuffled_minus_Original"] = (
    paired_2c["Shuffled positions"] - paired_2c["Original"]
)

paired_2c["Original_minus_Shuffled"] = (
    paired_2c["Original"] - paired_2c["Shuffled positions"]
)

paired_2c

In [ ]:
import matplotlib.pyplot as plt

plot_2c = track2c_summary_df.copy()

plt.figure(figsize=(6, 4))

plt.bar(
    plot_2c["Condition"],
    plot_2c["Mean_F1"],
    yerr=plot_2c["Std_F1"],
    capsize=5
)

plt.ylabel("Weighted F1 Score")
plt.title("Track 2C: Original vs Shuffled Position Structure")
plt.ylim(0.55, 0.78)

plt.show()

In [ ]:
from itertools import permutations
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import f1_score

def apply_global_position_permutation(X, perm):
    """
    Apply one fixed global permutation of the 4 CAR-T position blocks.

    perm tells us which old position goes into each new position.

    Example:
    perm = (1, 0, 2, 3)
    means:
    new Position 1 gets old Position 2
    new Position 2 gets old Position 1
    new Position 3 gets old Position 3
    new Position 4 gets old Position 4
    """

    X_df = X.copy().reset_index(drop=True)
    X_perm = X_df.copy()

    measurement_block_starts = [0, 60, 120]
    position_size = 15
    num_positions = 4

    for block_start in measurement_block_starts:
        original_blocks = []

        for old_pos in range(num_positions):
            start = block_start + old_pos * position_size
            end = start + position_size
            original_blocks.append(X_df.iloc[:, start:end].to_numpy())

        for new_pos, old_pos in enumerate(perm):
            start = block_start + new_pos * position_size
            end = start + position_size
            X_perm.iloc[:, start:end] = original_blocks[old_pos]

    return X_perm


# Quick test
test_perm = (1, 0, 2, 3)
X_test_perm = apply_global_position_permutation(X_quantum_all, test_perm)

print("Original shape:", X_quantum_all.shape)
print("Permuted shape:", X_test_perm.shape)

In [ ]:
# Compare Position 2 vs Position 3 on the exact same random splits

paired_pos23 = resample_results_df.pivot(
    index="Split",
    columns="Position",
    values="F1"
).reset_index()

paired_pos23["P2_minus_P3"] = paired_pos23["Position 2"] - paired_pos23["Position 3"]

print("Mean P2 - P3 gap:", round(paired_pos23["P2_minus_P3"].mean(), 4))
print("Std P2 - P3 gap:", round(paired_pos23["P2_minus_P3"].std(), 4))
print("Position 2 > Position 3:", (paired_pos23["P2_minus_P3"] > 0).sum(), "out of", len(paired_pos23))
print("Position 3 > Position 2:", (paired_pos23["P2_minus_P3"] < 0).sum(), "out of", len(paired_pos23))

paired_pos23

In [ ]:
import numpy as np
import pandas as pd

# Combine official train/test classical data
X_classical_all = np.vstack([
    np.asarray(train_data),
    np.asarray(test_data)
])

y_all = np.concatenate([train_labels, test_labels])

# Convert each 15-column position block back into motif ID
# Motif IDs will be 1 through 15, where 14 is terminal and 15 is empty if using the prompt's convention.

motif_df = pd.DataFrame()

for pos in range(4):
    start = pos * 15
    end = start + 15
    
    # argmax finds which motif category is active in that position
    motif_df[f"Position {pos + 1}"] = np.argmax(X_classical_all[:, start:end], axis=1) + 1

motif_df["Label"] = np.where(y_all == 1, "HIGH", "LOW")

motif_df.head()

In [ ]:
# Overall HIGH rate in the full dataset
overall_high_rate = (motif_df["Label"] == "HIGH").mean()
print("Overall HIGH rate:", round(overall_high_rate, 4))

def motif_high_rate_table(position_col):
    """
    For one position, compute:
    - how many times each motif appears
    - how many are HIGH
    - how many are LOW
    - HIGH rate
    - enrichment above/below overall HIGH rate
    """
    
    table = (
        motif_df
        .groupby(position_col)["Label"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )
    
    if "HIGH" not in table.columns:
        table["HIGH"] = 0
    if "LOW" not in table.columns:
        table["LOW"] = 0
    
    table["Total"] = table["HIGH"] + table["LOW"]
    table["HIGH_rate"] = table["HIGH"] / table["Total"]
    table["Enrichment_vs_overall"] = table["HIGH_rate"] - overall_high_rate
    
    table = table.sort_values("Enrichment_vs_overall", ascending=False)
    
    return table

pos2_motif_table = motif_high_rate_table("Position 2")
pos3_motif_table = motif_high_rate_table("Position 3")

print("Position 2 motif table:")
display(pos2_motif_table)

print("Position 3 motif table:")
display(pos3_motif_table)

In [ ]:
import numpy as np
import pandas as pd

position_cols = ["Position 1", "Position 2", "Position 3", "Position 4"]

def entropy_from_counts(counts):
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -(probs * np.log2(probs)).sum()

variation_rows = []

for pos_col in position_cols:
    counts = motif_df[pos_col].value_counts().sort_index()
    n = len(motif_df)

    terminal_count = counts.get(14, 0)
    empty_count = counts.get(15, 0)
    terminal_empty_count = terminal_count + empty_count

    dominant_motif = counts.idxmax()
    dominant_count = counts.max()

    entropy = entropy_from_counts(counts)
    max_entropy = np.log2(15)

    terminal_empty_mask = motif_df[pos_col].isin([14, 15])
    non_terminal_empty_mask = ~terminal_empty_mask

    terminal_empty_high_rate = (
        motif_df.loc[terminal_empty_mask, "Label"].eq("HIGH").mean()
        if terminal_empty_mask.sum() > 0 else np.nan
    )

    non_terminal_empty_high_rate = (
        motif_df.loc[non_terminal_empty_mask, "Label"].eq("HIGH").mean()
        if non_terminal_empty_mask.sum() > 0 else np.nan
    )

    variation_rows.append({
        "Position": pos_col,
        "Unique motifs used": counts.size,
        "Terminal count M14": terminal_count,
        "Empty count M15": empty_count,
        "Terminal/empty rate": terminal_empty_count / n,
        "Dominant motif": dominant_motif,
        "Dominant motif rate": dominant_count / n,
        "Entropy": entropy,
        "Normalized entropy": entropy / max_entropy,
        "HIGH rate if terminal/empty": terminal_empty_high_rate,
        "HIGH rate if not terminal/empty": non_terminal_empty_high_rate
    })

variation_summary_df = pd.DataFrame(variation_rows)

variation_summary_df.round(4)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# -----------------------------
# 1. Combine official train/test classical data
# -----------------------------

X_classical_all = pd.DataFrame(
    np.vstack([
        np.asarray(train_data),
        np.asarray(test_data)
    ])
)

y_all = np.concatenate([train_labels, test_labels])

print("X_classical_all:", X_classical_all.shape)
print("y_all:", y_all.shape)

# -----------------------------
# 2. Classical position slices
# -----------------------------
# Classical data has 60 features:
# Position 1 = columns 0-14
# Position 2 = columns 15-29
# Position 3 = columns 30-44
# Position 4 = columns 45-59

classical_position_slices = {
    "Position 1": list(range(0, 15)),
    "Position 2": list(range(15, 30)),
    "Position 3": list(range(30, 45)),
    "Position 4": list(range(45, 60)),
}

# -----------------------------
# 3. SVM grid
# -----------------------------

param_grid_classical_pos = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

# -----------------------------
# 4. Random train/test resampling
# -----------------------------

n_splits_classical = 20

splitter_classical = StratifiedShuffleSplit(
    n_splits=n_splits_classical,
    test_size=0.30,
    random_state=42
)

classical_position_results = []

for split_id, (train_idx, test_idx) in enumerate(splitter_classical.split(X_classical_all, y_all), start=1):
    print(f"\nClassical split {split_id}/{n_splits_classical}")

    y_train_split = y_all[train_idx]
    y_test_split = y_all[test_idx]

    for position_name, cols in classical_position_slices.items():

        X_train_pos = X_classical_all.iloc[train_idx, cols]
        X_test_pos = X_classical_all.iloc[test_idx, cols]

        svm = GridSearchCV(
            SVC(kernel="rbf"),
            param_grid_classical_pos,
            cv=5,
            scoring="f1_weighted",
            n_jobs=-1
        )

        svm.fit(X_train_pos, y_train_split)

        preds = svm.predict(X_test_pos)
        f1 = f1_score(y_test_split, preds, average="weighted")

        classical_position_results.append({
            "Split": split_id,
            "Position": position_name,
            "F1": f1,
            "Best C": svm.best_params_["C"],
            "Best gamma": svm.best_params_["gamma"]
        })

# -----------------------------
# 5. Summarize mean/std
# -----------------------------

classical_position_results_df = pd.DataFrame(classical_position_results)

classical_position_summary_df = (
    classical_position_results_df
    .groupby("Position")
    .agg(
        Mean_F1=("F1", "mean"),
        Std_F1=("F1", "std"),
        Min_F1=("F1", "min"),
        Max_F1=("F1", "max")
    )
    .reset_index()
)

classical_position_summary_df["Mean_F1"] = classical_position_summary_df["Mean_F1"].round(4)
classical_position_summary_df["Std_F1"] = classical_position_summary_df["Std_F1"].round(4)
classical_position_summary_df["Min_F1"] = classical_position_summary_df["Min_F1"].round(4)
classical_position_summary_df["Max_F1"] = classical_position_summary_df["Max_F1"].round(4)

classical_position_summary_df = classical_position_summary_df.sort_values("Mean_F1", ascending=False)

classical_position_summary_df

In [ ]:
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Convert to DataFrame to make column slicing easy
X_train_classical = pd.DataFrame(np.asarray(train_data))
X_test_classical = pd.DataFrame(np.asarray(test_data))

y_train = np.asarray(train_labels)
y_test = np.asarray(test_labels)

# Classical one-hot data has 60 columns:
# Position 1 = cols 0-14
# Position 2 = cols 15-29
# Position 3 = cols 30-44
# Position 4 = cols 45-59
classical_position_slices = {
    "Position 1": list(range(0, 15)),
    "Position 2": list(range(15, 30)),
    "Position 3": list(range(30, 45)),
    "Position 4": list(range(45, 60)),
}

param_grid_classical_pos = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

classical_one_split_results = []

for position_name, cols in classical_position_slices.items():
    print(f"Training classical SVM for {position_name}...")

    X_train_pos = X_train_classical.iloc[:, cols]
    X_test_pos = X_test_classical.iloc[:, cols]

    svm = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid_classical_pos,
        cv=10,
        scoring="f1_weighted",
        n_jobs=-1
    )

    svm.fit(X_train_pos, y_train)

    preds = svm.predict(X_test_pos)
    f1 = f1_score(y_test, preds, average="weighted")

    classical_one_split_results.append({
        "Position": position_name,
        "Features Used": len(cols),
        "Best C": svm.best_params_["C"],
        "Best gamma": svm.best_params_["gamma"],
        "Test F1": round(f1, 4)
    })

classical_one_split_df = pd.DataFrame(classical_one_split_results)
classical_one_split_df = classical_one_split_df.sort_values("Test F1", ascending=False)

classical_one_split_df

In [ ]:
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Quantum projected data already has 180 columns:
# X block: columns 0-59
# Y block: columns 60-119
# Z block: columns 120-179
X_train_quantum = projections_train
X_test_quantum = projections_test

y_train = np.asarray(train_labels)
y_test = np.asarray(test_labels)

# Each CAR-T position has 15 features inside each measurement block.
# So each position uses 15 X + 15 Y + 15 Z = 45 quantum features.
quantum_position_slices = {
    "Position 1": list(range(0, 15)) + list(range(60, 75)) + list(range(120, 135)),
    "Position 2": list(range(15, 30)) + list(range(75, 90)) + list(range(135, 150)),
    "Position 3": list(range(30, 45)) + list(range(90, 105)) + list(range(150, 165)),
    "Position 4": list(range(45, 60)) + list(range(105, 120)) + list(range(165, 180)),
}

param_grid_quantum_pos = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

quantum_one_split_results = []

for position_name, cols in quantum_position_slices.items():
    print(f"Training quantum SVM for {position_name}...")

    X_train_pos = X_train_quantum.iloc[:, cols]
    X_test_pos = X_test_quantum.iloc[:, cols]

    svm = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid_quantum_pos,
        cv=10,
        scoring="f1_weighted",
        n_jobs=-1
    )

    svm.fit(X_train_pos, y_train)

    preds = svm.predict(X_test_pos)
    f1 = f1_score(y_test, preds, average="weighted")

    quantum_one_split_results.append({
        "Position": position_name,
        "Features Used": len(cols),
        "Best C": svm.best_params_["C"],
        "Best gamma": svm.best_params_["gamma"],
        "Test F1": round(f1, 4)
    })

quantum_one_split_df = pd.DataFrame(quantum_one_split_results)
quantum_one_split_df = quantum_one_split_df.sort_values("Test F1", ascending=False)

quantum_one_split_df

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# -----------------------------
# 1. Combine official train/test quantum data
# -----------------------------

X_quantum_all = pd.concat(
    [projections_train, projections_test],
    axis=0
).reset_index(drop=True)

y_all = np.concatenate([train_labels, test_labels])

print("X_quantum_all:", X_quantum_all.shape)
print("y_all:", y_all.shape)

# -----------------------------
# 2. Quantum position slices
# -----------------------------
# projections are laid out as:
# X block: columns 0-59
# Y block: columns 60-119
# Z block: columns 120-179
#
# Each position gets 15 features per block.
# So each position uses 15 X + 15 Y + 15 Z = 45 features.

quantum_position_slices = {
    "Position 1": list(range(0, 15)) + list(range(60, 75)) + list(range(120, 135)),
    "Position 2": list(range(15, 30)) + list(range(75, 90)) + list(range(135, 150)),
    "Position 3": list(range(30, 45)) + list(range(90, 105)) + list(range(150, 165)),
    "Position 4": list(range(45, 60)) + list(range(105, 120)) + list(range(165, 180)),
}

# -----------------------------
# 3. SVM hyperparameter grid
# -----------------------------

param_grid_quantum_pos = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

# -----------------------------
# 4. Repeated random train/test splits
# -----------------------------

n_splits_quantum = 20

splitter_quantum = StratifiedShuffleSplit(
    n_splits=n_splits_quantum,
    test_size=0.30,
    random_state=42
)

quantum_resample_results = []

for split_id, (train_idx, test_idx) in enumerate(splitter_quantum.split(X_quantum_all, y_all), start=1):
    print(f"\nQuantum split {split_id}/{n_splits_quantum}")

    y_train_split = y_all[train_idx]
    y_test_split = y_all[test_idx]

    for position_name, cols in quantum_position_slices.items():
        print(f"  Training {position_name}...")

        X_train_pos = X_quantum_all.iloc[train_idx, cols]
        X_test_pos = X_quantum_all.iloc[test_idx, cols]

        svm = GridSearchCV(
            SVC(kernel="rbf"),
            param_grid_quantum_pos,
            cv=5,
            scoring="f1_weighted",
            n_jobs=-1
        )

        svm.fit(X_train_pos, y_train_split)

        preds = svm.predict(X_test_pos)
        f1 = f1_score(y_test_split, preds, average="weighted")

        quantum_resample_results.append({
            "Split": split_id,
            "Position": position_name,
            "F1": f1,
            "Best C": svm.best_params_["C"],
            "Best gamma": svm.best_params_["gamma"]
        })

# -----------------------------
# 5. Summary table: mean/std/min/max F1
# -----------------------------

quantum_resample_results_df = pd.DataFrame(quantum_resample_results)

quantum_resample_summary_df = (
    quantum_resample_results_df
    .groupby("Position")
    .agg(
        Mean_F1=("F1", "mean"),
        Std_F1=("F1", "std"),
        Min_F1=("F1", "min"),
        Max_F1=("F1", "max")
    )
    .reset_index()
)

quantum_resample_summary_df["Mean_F1"] = quantum_resample_summary_df["Mean_F1"].round(4)
quantum_resample_summary_df["Std_F1"] = quantum_resample_summary_df["Std_F1"].round(4)
quantum_resample_summary_df["Min_F1"] = quantum_resample_summary_df["Min_F1"].round(4)
quantum_resample_summary_df["Max_F1"] = quantum_resample_summary_df["Max_F1"].round(4)

quantum_resample_summary_df = quantum_resample_summary_df.sort_values("Mean_F1", ascending=False)

quantum_resample_summary_df

In [ ]:
# ============================================
# Table 1: One official split comparison
# Classical vs Quantum by position
# ============================================

one_split_comparison_df = classical_one_split_df[
    ["Position", "Features Used", "Best C", "Best gamma", "Test F1"]
].rename(columns={
    "Features Used": "Classical Features",
    "Best C": "Classical Best C",
    "Best gamma": "Classical Best gamma",
    "Test F1": "Classical Test F1"
}).merge(
    quantum_one_split_df[
        ["Position", "Features Used", "Best C", "Best gamma", "Test F1"]
    ].rename(columns={
        "Features Used": "Quantum Features",
        "Best C": "Quantum Best C",
        "Best gamma": "Quantum Best gamma",
        "Test F1": "Quantum Test F1"
    }),
    on="Position"
)

one_split_comparison_df["Quantum - Classical"] = (
    one_split_comparison_df["Quantum Test F1"]
    - one_split_comparison_df["Classical Test F1"]
)

one_split_comparison_df["Position Number"] = (
    one_split_comparison_df["Position"].str.extract(r"(\d+)").astype(int)
)

one_split_comparison_df = (
    one_split_comparison_df
    .sort_values("Position Number")
    .drop(columns=["Position Number"])
    .round(4)
)

print("One Official Train/Test Split: Classical vs Quantum")
display(one_split_comparison_df)


# ============================================
# Table 2: 20 random resamples comparison
# Classical vs Quantum by position, with median
# ============================================

classical_resample_summary_with_median = (
    classical_position_results_df
    .groupby("Position")
    .agg(
        Classical_Mean_F1=("F1", "mean"),
        Classical_Median_F1=("F1", "median"),
        Classical_Std_F1=("F1", "std"),
        Classical_Min_F1=("F1", "min"),
        Classical_Max_F1=("F1", "max")
    )
    .reset_index()
)

quantum_resample_summary_with_median = (
    quantum_resample_results_df
    .groupby("Position")
    .agg(
        Quantum_Mean_F1=("F1", "mean"),
        Quantum_Median_F1=("F1", "median"),
        Quantum_Std_F1=("F1", "std"),
        Quantum_Min_F1=("F1", "min"),
        Quantum_Max_F1=("F1", "max")
    )
    .reset_index()
)

resample_comparison_df = classical_resample_summary_with_median.merge(
    quantum_resample_summary_with_median,
    on="Position"
)

resample_comparison_df["Quantum - Classical Mean F1"] = (
    resample_comparison_df["Quantum_Mean_F1"]
    - resample_comparison_df["Classical_Mean_F1"]
)

resample_comparison_df["Quantum - Classical Median F1"] = (
    resample_comparison_df["Quantum_Median_F1"]
    - resample_comparison_df["Classical_Median_F1"]
)

resample_comparison_df["Position Number"] = (
    resample_comparison_df["Position"].str.extract(r"(\d+)").astype(int)
)

resample_comparison_df = (
    resample_comparison_df
    .sort_values("Position Number")
    .drop(columns=["Position Number"])
    .round(4)
)

print("20 Random Resamples: Classical vs Quantum with Median")
display(resample_comparison_df)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import f1_score

# -----------------------------
# 1. Convert inputs safely
# -----------------------------

def to_dataframe(X):
    """
    Convert numpy array or pandas DataFrame into a clean DataFrame.
    """
    if isinstance(X, pd.DataFrame):
        return X.reset_index(drop=True)
    return pd.DataFrame(np.asarray(X))


X_train_classical = to_dataframe(train_data)
X_test_classical = to_dataframe(test_data)

X_train_quantum = to_dataframe(projections_train)
X_test_quantum = to_dataframe(projections_test)

y_train = np.asarray(train_labels)
y_test = np.asarray(test_labels)

print("Classical train:", X_train_classical.shape)
print("Classical test:", X_test_classical.shape)
print("Quantum train:", X_train_quantum.shape)
print("Quantum test:", X_test_quantum.shape)
print("Train labels:", y_train.shape)
print("Test labels:", y_test.shape)


# -----------------------------
# 2. Basic sanity checks
# -----------------------------

assert X_train_classical.shape == (172, 60), "Expected classical train shape (172, 60)"
assert X_test_classical.shape == (74, 60), "Expected classical test shape (74, 60)"

assert X_train_quantum.shape == (172, 180), "Expected quantum train shape (172, 180)"
assert X_test_quantum.shape == (74, 180), "Expected quantum test shape (74, 180)"

assert y_train.shape == (172,), "Expected train labels shape (172,)"
assert y_test.shape == (74,), "Expected test labels shape (74,)"

print("All shape checks passed.")


# -----------------------------
# 3. Position slices
# -----------------------------
# Classical:
# 60 features = 4 positions × 15 motif categories
#
# Quantum:
# 180 features = 3 measurement blocks × 60 features
# Each position uses 15 features from each block = 45 features total.

classical_position_slices = {
    "Position 1": list(range(0, 15)),
    "Position 2": list(range(15, 30)),
    "Position 3": list(range(30, 45)),
    "Position 4": list(range(45, 60)),
}

quantum_position_slices = {
    "Position 1": list(range(0, 15)) + list(range(60, 75)) + list(range(120, 135)),
    "Position 2": list(range(15, 30)) + list(range(75, 90)) + list(range(135, 150)),
    "Position 3": list(range(30, 45)) + list(range(90, 105)) + list(range(150, 165)),
    "Position 4": list(range(45, 60)) + list(range(105, 120)) + list(range(165, 180)),
}

print("\nClassical position feature counts:")
for pos, cols in classical_position_slices.items():
    print(pos, len(cols))

print("\nQuantum position feature counts:")
for pos, cols in quantum_position_slices.items():
    print(pos, len(cols))


# -----------------------------
# 4. Slice sanity checks
# -----------------------------

for pos, cols in classical_position_slices.items():
    assert len(cols) == 15, f"{pos} classical should have 15 features"

for pos, cols in quantum_position_slices.items():
    assert len(cols) == 45, f"{pos} quantum should have 45 features"

print("\nAll position slice checks passed.")

In [ ]:
# -----------------------------
# 5. Reusable SVM training function
# -----------------------------

param_grid = {
    "C": [0.1, 1, 5, 10, 20],
    "gamma": [0.001, 0.005, 0.01, 0.05, 0.1]
}

def train_position_svm(
    X_train,
    X_test,
    y_train,
    y_test,
    cols,
    representation_name,
    position_name,
    cv=10
):
    """
    Train an RBF-SVM on one position's features only.

    X_train, X_test:
        Full feature matrices.

    cols:
        Columns corresponding to one CAR-T position.

    representation_name:
        "Classical" or "Quantum"

    position_name:
        "Position 1", "Position 2", etc.
    """

    X_train_pos = X_train.iloc[:, cols]
    X_test_pos = X_test.iloc[:, cols]

    model = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1
    )

    model.fit(X_train_pos, y_train)

    preds = model.predict(X_test_pos)
    f1 = f1_score(y_test, preds, average="weighted")

    return {
        "Representation": representation_name,
        "Position": position_name,
        "Features Used": len(cols),
        "Best C": model.best_params_["C"],
        "Best gamma": model.best_params_["gamma"],
        "Test F1": round(f1, 4)
    }

In [ ]:
# -----------------------------
# 6. Test helper function on Position 1
# -----------------------------

test_results = []

test_results.append(
    train_position_svm(
        X_train_classical,
        X_test_classical,
        y_train,
        y_test,
        classical_position_slices["Position 1"],
        "Classical",
        "Position 1",
        cv=10
    )
)

test_results.append(
    train_position_svm(
        X_train_quantum,
        X_test_quantum,
        y_train,
        y_test,
        quantum_position_slices["Position 1"],
        "Quantum",
        "Position 1",
        cv=10
    )
)

pd.DataFrame(test_results)

In [ ]:
# -----------------------------
# 7. Official train/test split:
# Classical vs Quantum for all positions
# -----------------------------

official_split_results = []

for position_name in ["Position 1", "Position 2", "Position 3", "Position 4"]:
    
    # Classical
    official_split_results.append(
        train_position_svm(
            X_train_classical,
            X_test_classical,
            y_train,
            y_test,
            classical_position_slices[position_name],
            "Classical",
            position_name,
            cv=10
        )
    )
    
    # Quantum
    official_split_results.append(
        train_position_svm(
            X_train_quantum,
            X_test_quantum,
            y_train,
            y_test,
            quantum_position_slices[position_name],
            "Quantum",
            position_name,
            cv=10
        )
    )

official_split_results_df = pd.DataFrame(official_split_results)

official_split_results_df

In [ ]:
official_comparison_df = official_split_results_df.pivot(
    index="Position",
    columns="Representation",
    values="Test F1"
).reset_index()

official_comparison_df["Quantum - Classical"] = (
    official_comparison_df["Quantum"] - official_comparison_df["Classical"]
)

official_comparison_df["Position Number"] = (
    official_comparison_df["Position"].str.extract(r"(\d+)").astype(int)
)

official_comparison_df = (
    official_comparison_df
    .sort_values("Position Number")
    .drop(columns=["Position Number"])
    .round(4)
)

official_comparison_df

# Hardware Noise Resilience Across Measurement Bases

This section follows the implementation plan for testing X, Y, and Z measurement-basis resilience using classical baselines, per-basis SVMs, boundary support-vector samples, ideal simulation, and QPU hardware runs.

In [ ]:
# ============================================================
# Phase 1.1–1.2: Verify loaded data and slice X/Y/Z projections
# ============================================================

import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# -----------------------------
# Helper: make sure data is in DataFrame form
# -----------------------------

def to_dataframe(X):
    """
    Convert numpy array or pandas DataFrame into a clean pandas DataFrame.
    """
    if isinstance(X, pd.DataFrame):
        return X.reset_index(drop=True)
    return pd.DataFrame(np.asarray(X))


# -----------------------------
# Phase 1.1: Verify loaded data
# -----------------------------

X_train_classical = to_dataframe(train_data)
X_test_classical = to_dataframe(test_data)

X_train_quantum = to_dataframe(projections_train)
X_test_quantum = to_dataframe(projections_test)

y_train = np.asarray(train_labels)
y_test = np.asarray(test_labels)

print("X_train_classical:", X_train_classical.shape)
print("X_test_classical:", X_test_classical.shape)
print("X_train_quantum:", X_train_quantum.shape)
print("X_test_quantum:", X_test_quantum.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

assert X_train_classical.shape == (172, 60), "Expected train_data shape (172, 60)"
assert X_test_classical.shape == (74, 60), "Expected test_data shape (74, 60)"
assert X_train_quantum.shape == (172, 180), "Expected projections_train shape (172, 180)"
assert X_test_quantum.shape == (74, 180), "Expected projections_test shape (74, 180)"
assert y_train.shape == (172,), "Expected train_labels shape (172,)"
assert y_test.shape == (74,), "Expected test_labels shape (74,)"

print("\nData checks passed.")


# -----------------------------
# Phase 1.2: Slice quantum projections by measurement basis
# Following your implementation plan:
#
# X basis = columns 0:60
# Y basis = columns 60:120
# Z basis = columns 120:180
# -----------------------------

basis_X_train = X_train_quantum.iloc[:, 0:60]
basis_X_test = X_test_quantum.iloc[:, 0:60]

basis_Y_train = X_train_quantum.iloc[:, 60:120]
basis_Y_test = X_test_quantum.iloc[:, 60:120]

basis_Z_train = X_train_quantum.iloc[:, 120:180]
basis_Z_test = X_test_quantum.iloc[:, 120:180]

print("\nBasis shapes:")
print("basis_X_train:", basis_X_train.shape)
print("basis_X_test:", basis_X_test.shape)
print("basis_Y_train:", basis_Y_train.shape)
print("basis_Y_test:", basis_Y_test.shape)
print("basis_Z_train:", basis_Z_train.shape)
print("basis_Z_test:", basis_Z_test.shape)

assert basis_X_train.shape == (172, 60)
assert basis_X_test.shape == (74, 60)
assert basis_Y_train.shape == (172, 60)
assert basis_Y_test.shape == (74, 60)
assert basis_Z_train.shape == (172, 60)
assert basis_Z_test.shape == (74, 60)

print("\nBasis slicing checks passed.")

In [ ]:
# ============================================================
# Phase 1.3–1.5: Train classical, full quantum, and per-basis SVMs
# ============================================================

# We use one reusable function so every model is trained the same way.

svm_param_grid = {
    "C": [0.1, 0.5, 1, 2, 5, 8.5, 10, 10.75, 20, 50],
    "gamma": [0.001, 0.005, 0.01, 0.02, 0.04, 0.05, 0.1]
}

def train_and_evaluate_svm(model_name, X_train, X_test, y_train, y_test, cv=10):
    """
    Train an RBF-SVM with GridSearchCV and return:
    - fitted GridSearchCV model
    - weighted F1 score
    - best hyperparameters
    """
    
    model = GridSearchCV(
        SVC(kernel="rbf"),
        svm_param_grid,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
        refit=True
    )
    
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    f1 = f1_score(y_test, preds, average="weighted")
    
    result = {
        "Model": model_name,
        "Features Used": X_train.shape[1],
        "Best C": model.best_params_["C"],
        "Best gamma": model.best_params_["gamma"],
        "Weighted F1": round(f1, 4)
    }
    
    return model, result


# -----------------------------
# Train all Phase 1 models
# -----------------------------

phase1_models = {}
phase1_results = []

# 1.3 Classical 60-dim baseline
model_classical, result_classical = train_and_evaluate_svm(
    "Classical 60-dim",
    X_train_classical,
    X_test_classical,
    y_train,
    y_test,
    cv=10
)

phase1_models["Classical 60-dim"] = model_classical
phase1_results.append(result_classical)


# 1.4 Full quantum 180-dim baseline
model_full_quantum, result_full_quantum = train_and_evaluate_svm(
    "Full Quantum 180-dim",
    X_train_quantum,
    X_test_quantum,
    y_train,
    y_test,
    cv=10
)

phase1_models["Full Quantum 180-dim"] = model_full_quantum
phase1_results.append(result_full_quantum)


# 1.5 X-only model
model_X, result_X = train_and_evaluate_svm(
    "X-only Quantum",
    basis_X_train,
    basis_X_test,
    y_train,
    y_test,
    cv=10
)

phase1_models["X-only Quantum"] = model_X
phase1_results.append(result_X)


# 1.5 Y-only model
model_Y, result_Y = train_and_evaluate_svm(
    "Y-only Quantum",
    basis_Y_train,
    basis_Y_test,
    y_train,
    y_test,
    cv=10
)

phase1_models["Y-only Quantum"] = model_Y
phase1_results.append(result_Y)


# 1.5 Z-only model
model_Z, result_Z = train_and_evaluate_svm(
    "Z-only Quantum",
    basis_Z_train,
    basis_Z_test,
    y_train,
    y_test,
    cv=10
)

phase1_models["Z-only Quantum"] = model_Z
phase1_results.append(result_Z)


# -----------------------------
# Display clean results table
# -----------------------------

phase1_f1_df = pd.DataFrame(phase1_results)

phase1_f1_df

In [ ]:
# ============================================================
# Phase 1.6: Select boundary support-vector samples
# ============================================================

# Get the best fitted classical SVM from GridSearchCV
best_classical_svm = model_classical.best_estimator_

# support_ gives row indices, relative to X_train_classical
support_indices = best_classical_svm.support_

print("Number of classical support vectors:", len(support_indices))

# Get labels and decision scores for all training samples
train_decision_scores = best_classical_svm.decision_function(X_train_classical)

support_df = pd.DataFrame({
    "train_index": support_indices,
    "label": y_train[support_indices],
    "decision_score": train_decision_scores[support_indices],
    "abs_decision_score": np.abs(train_decision_scores[support_indices])
})

# Smaller absolute decision score = closer to the decision boundary
support_df = support_df.sort_values("abs_decision_score")

print("Support-vector label counts:")
display(support_df["label"].value_counts())

display(support_df.head(10))

In [ ]:
# ============================================================
# Phase 1.6 continued: Select 5 positive and 5 negative boundary samples
# ============================================================

# Sort support vectors by closeness to boundary
support_df_sorted = support_df.sort_values("abs_decision_score")

# Select 5 support vectors from each class
positive_boundary = support_df_sorted[support_df_sorted["label"] == 1.0].head(5)
negative_boundary = support_df_sorted[support_df_sorted["label"] == -1.0].head(5)

boundary_samples_df = pd.concat(
    [positive_boundary, negative_boundary],
    axis=0
).sort_values("abs_decision_score").reset_index(drop=True)

print("Selected boundary samples:")
display(boundary_samples_df)

print("Label counts:")
display(boundary_samples_df["label"].value_counts())

# Save the selected training indices
boundary_train_indices = boundary_samples_df["train_index"].to_numpy()

# Extract the corresponding 60-dim classical features
boundary_classical_features = X_train_classical.iloc[boundary_train_indices, :].reset_index(drop=True)
boundary_labels = y_train[boundary_train_indices]

print("boundary_train_indices:", boundary_train_indices)
print("boundary_classical_features shape:", boundary_classical_features.shape)
print("boundary_labels shape:", boundary_labels.shape)

display(boundary_classical_features.head())

In [ ]:
# ============================================================
# Phase 2.1: Initialize 60-qubit ZZFeatureMap circuit template
# ============================================================

from qiskit.circuit.library import ZZFeatureMap

num_qubits = 60
feature_dimension = 60
reps = 2

zz_feature_map = ZZFeatureMap(
    feature_dimension=feature_dimension,
    reps=reps
)

print("ZZFeatureMap created.")
print("Number of qubits:", zz_feature_map.num_qubits)
print("Number of parameters:", len(zz_feature_map.parameters))
print("Circuit depth:", zz_feature_map.decompose().depth())

assert zz_feature_map.num_qubits == 60
assert len(zz_feature_map.parameters) == 60

print("ZZFeatureMap checks passed.")

In [ ]:
# ============================================================
# Phase 2.2: Define 180 observables
# 60 X observables, 60 Y observables, 60 Z observables
# ============================================================

from qiskit.quantum_info import SparsePauliOp

def single_qubit_pauli_observable(pauli, qubit_index, num_qubits=60):
    """
    Create a SparsePauliOp for measuring one Pauli operator
    on one qubit and identity on all other qubits.

    Important Qiskit convention:
    Pauli label strings are ordered with qubit 0 on the RIGHT.
    So to target qubit_index q, we place the Pauli at:
        label[num_qubits - 1 - q]
    """
    label = ["I"] * num_qubits
    label[num_qubits - 1 - qubit_index] = pauli
    label = "".join(label)
    
    return SparsePauliOp.from_list([(label, 1.0)])


# Create basis-specific observable lists
observables_X = [
    single_qubit_pauli_observable("X", q, num_qubits)
    for q in range(num_qubits)
]

observables_Y = [
    single_qubit_pauli_observable("Y", q, num_qubits)
    for q in range(num_qubits)
]

observables_Z = [
    single_qubit_pauli_observable("Z", q, num_qubits)
    for q in range(num_qubits)
]

# Combine into one list in the same order as your implementation plan:
# X columns 0-59, Y columns 60-119, Z columns 120-179
observables_all = observables_X + observables_Y + observables_Z

print("Number of X observables:", len(observables_X))
print("Number of Y observables:", len(observables_Y))
print("Number of Z observables:", len(observables_Z))
print("Total observables:", len(observables_all))

assert len(observables_X) == 60
assert len(observables_Y) == 60
assert len(observables_Z) == 60
assert len(observables_all) == 180

print("Observable checks passed.")

In [ ]:
# Inspect a few Pauli labels to verify ordering

print("X on qubit 0:")
print(observables_X[0].paulis[0])

print("\nX on qubit 1:")
print(observables_X[1].paulis[0])

print("\nZ on qubit 59:")
print(observables_Z[59].paulis[0])

In [ ]:
# ============================================================
# Phase 2.3: Bind parameters for the 10 boundary samples
# ============================================================

import re

def parameter_index(param):
    """
    Extract the numeric index from a Qiskit parameter like x[0], x[1], ...
    """
    match = re.search(r"\[(\d+)\]", param.name)
    if match is None:
        raise ValueError(f"Could not parse parameter index from {param.name}")
    return int(match.group(1))


# Sort parameters as x[0], x[1], ..., x[59]
zz_params = sorted(
    list(zz_feature_map.parameters),
    key=parameter_index
)

print("First 5 parameters:", zz_params[:5])
print("Last 5 parameters:", zz_params[-5:])
print("Number of parameters:", len(zz_params))

assert len(zz_params) == 60


# Create one bound circuit per boundary sample
boundary_circuits = []

for i in range(boundary_classical_features.shape[0]):
    feature_vector = boundary_classical_features.iloc[i].to_numpy(dtype=float)

    assert feature_vector.shape == (60,)

    param_bindings = {
        zz_params[j]: feature_vector[j]
        for j in range(60)
    }

    bound_circuit = zz_feature_map.assign_parameters(
        param_bindings,
        inplace=False
    )

    boundary_circuits.append(bound_circuit)

print("Number of bound circuits:", len(boundary_circuits))
print("Circuit 0 parameters remaining:", len(boundary_circuits[0].parameters))
print("Circuit 0 qubits:", boundary_circuits[0].num_qubits)

assert len(boundary_circuits) == 10
assert len(boundary_circuits[0].parameters) == 0
assert boundary_circuits[0].num_qubits == 60

print("Parameter binding checks passed.")

In [ ]:
# ============================================================
# Phase 2.4A: Connect to IBM Runtime and select backend
# ============================================================

from qiskit_ibm_runtime import QiskitRuntimeService

# Use your saved IBM Quantum account.
# This should work if Start_From_Here.ipynb already saved/loaded your token.
service = QiskitRuntimeService()

# List available real quantum hardware backends
available_backends = service.backends(simulator=False, operational=True)

print("Available operational QPU backends:")
for backend in available_backends:
    status = backend.status()
    print(
        backend.name,
        "| qubits:", backend.num_qubits,
        "| pending jobs:", status.pending_jobs
    )

# Choose a backend with at least 60 qubits and the fewest pending jobs
candidate_backends = [
    backend for backend in available_backends
    if backend.num_qubits >= 60
]

assert len(candidate_backends) > 0, "No operational backend with at least 60 qubits found."

backend = min(
    candidate_backends,
    key=lambda b: b.status().pending_jobs
)

print("\nSelected backend:", backend.name)
print("Number of qubits:", backend.num_qubits)
print("Pending jobs:", backend.status().pending_jobs)

In [ ]:
# ============================================================
# Phase 2.4B: Transpile circuits and observables to backend ISA
# ============================================================

try:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
except ImportError:
    from qiskit.transpiler import generate_preset_pass_manager

# Use optimization level 1 first because it is faster and safer for hackathon iteration.
optimization_level = 1

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=optimization_level
)

print("Pass manager created.")
print("Backend:", backend.name)
print("Optimization level:", optimization_level)

# Transpile the 10 bound circuits to the backend instruction set architecture.
isa_circuits = pm.run(boundary_circuits)

print("\nNumber of ISA circuits:", len(isa_circuits))
print("ISA circuit 0 qubits:", isa_circuits[0].num_qubits)
print("ISA circuit 0 depth:", isa_circuits[0].depth())
print("ISA circuit 0 parameters remaining:", len(isa_circuits[0].parameters))

assert len(isa_circuits) == 10
assert len(isa_circuits[0].parameters) == 0

print("\nCircuit transpilation checks passed.")


# ------------------------------------------------------------
# Apply each transpiled circuit's layout to the observables.
#
# Important:
# After transpilation, logical qubits may be mapped to different physical qubits.
# The observables must be transformed with the circuit layout so that
# "measure X on logical qubit 0" still targets the correct physical qubit.
# ------------------------------------------------------------

isa_observables_by_circuit = []

for circuit in isa_circuits:
    isa_obs_for_this_circuit = [
        obs.apply_layout(circuit.layout)
        for obs in observables_all
    ]
    isa_observables_by_circuit.append(isa_obs_for_this_circuit)

print("\nNumber of observable sets:", len(isa_observables_by_circuit))
print("Observables per circuit:", len(isa_observables_by_circuit[0]))

assert len(isa_observables_by_circuit) == 10
assert len(isa_observables_by_circuit[0]) == 180

print("Observable layout checks passed.")

In [ ]:
# ============================================================
# Phase 3.0: QPU smoke test
# 1 circuit, 3 observables: X0, Y0, Z0
# ============================================================

from qiskit_ibm_runtime import EstimatorV2 as Estimator
import numpy as np

# Use only the first transpiled circuit
smoke_circuit = isa_circuits[0]

# Use only 3 observables for qubit 0: X0, Y0, Z0
smoke_observables = [
    isa_observables_by_circuit[0][0],     # X on qubit 0
    isa_observables_by_circuit[0][60],    # Y on qubit 0
    isa_observables_by_circuit[0][120],   # Z on qubit 0
]

# Create EstimatorV2
estimator = Estimator(mode=backend)

# Raw/no mitigation first
estimator.options.resilience_level = 0

print("Submitting smoke test job...")

smoke_job = estimator.run(
    [(smoke_circuit, smoke_observables)],
    precision=0.05
)

print("Smoke test job ID:", smoke_job.job_id())

In [ ]:
# ============================================================
# FAST Smoke Test Block 1: Submit
# 1 circuit, 1 observable, low precision
# ============================================================

from qiskit_ibm_runtime import EstimatorV2 as Estimator
import numpy as np

# Use only the first transpiled circuit
fast_smoke_circuit = isa_circuits[0]

# Use only ONE observable: Z on qubit 0
# observables_all order: X 0-59, Y 60-119, Z 120-179
fast_smoke_observables = [
    isa_observables_by_circuit[0][120]   # Z on qubit 0
]

estimator_fast = Estimator(mode=backend)
estimator_fast.options.resilience_level = 0

print("Submitting FAST smoke test job...")

fast_smoke_job = estimator_fast.run(
    [(fast_smoke_circuit, fast_smoke_observables)],
    precision=0.20
)

fast_smoke_job_id = fast_smoke_job.job_id()

print("FAST smoke test job ID:", fast_smoke_job_id)

In [ ]:
# ============================================================
# FAST Smoke Test Block 2: Check status and retrieve only if done
# ============================================================

import numpy as np

# If your notebook lost the variable, paste the job ID manually:
# fast_smoke_job_id = "PASTE_JOB_ID_HERE"

fast_smoke_job_reloaded = service.job(fast_smoke_job_id)

status = fast_smoke_job_reloaded.status()
print("FAST smoke test job ID:", fast_smoke_job_reloaded.job_id())
print("Status:", status)

status_str = str(status).lower()

if "done" in status_str or "completed" in status_str:
    fast_smoke_result = fast_smoke_job_reloaded.result()
    fast_smoke_pub_result = fast_smoke_result[0]

    fast_smoke_evs = np.asarray(fast_smoke_pub_result.data.evs)

    print("FAST smoke expectation values:", fast_smoke_evs)
    print("Shape:", fast_smoke_evs.shape)

    assert fast_smoke_evs.shape == (1,)

    print("FAST smoke test retrieval passed.")

elif "cancel" in status_str or "fail" in status_str or "error" in status_str:
    print("Job did not complete successfully. Check details:")
    try:
        print(fast_smoke_job_reloaded.logs())
    except Exception as e:
        print("Could not fetch logs:", e)

else:
    print("Job is not done yet. Wait and rerun this cell in a few minutes.")

In [ ]:
# Re-check and retrieve fast smoke test result if finished

import numpy as np

fast_smoke_job_id = "d834rj7oha1c73bma3k0"
fast_smoke_job_reloaded = service.job(fast_smoke_job_id)

status = fast_smoke_job_reloaded.status()
print("FAST smoke test job ID:", fast_smoke_job_reloaded.job_id())
print("Status:", status)

status_str = str(status).lower()

if "done" in status_str or "completed" in status_str:
    fast_smoke_result = fast_smoke_job_reloaded.result()
    fast_smoke_pub_result = fast_smoke_result[0]

    fast_smoke_evs = np.asarray(fast_smoke_pub_result.data.evs)

    print("FAST smoke expectation values:", fast_smoke_evs)
    print("Shape:", fast_smoke_evs.shape)

elif "running" in status_str:
    print("Still running. Wait a few minutes and rerun this cell.")

elif "queued" in status_str:
    print("Still queued. Wait and rerun this cell.")

else:
    print("Unexpected status. Check job details.")

In [ ]:
fast_smoke_job_id = "d834rj7oha1c73bma3k0"

fast_job = service.job(fast_smoke_job_id)
print("Fast smoke job status before cancel:", fast_job.status())

fast_job.cancel()
print("Cancel requested.")

In [ ]:
# ============================================================
# Submit full QPU job directly
# 10 boundary circuits × 180 observables
# ============================================================

from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator_full = Estimator(mode=backend)
estimator_full.options.resilience_level = 0

# Each PUB = one circuit with its 180 layout-adjusted observables
full_pubs = [
    (isa_circuits[i], isa_observables_by_circuit[i])
    for i in range(len(isa_circuits))
]

print("Number of circuits/PUBs:", len(full_pubs))
print("Observables per circuit:", len(full_pubs[0][1]))

print("Submitting full QPU job...")

full_qpu_job = estimator_full.run(
    full_pubs,
    precision=0.20
)

full_qpu_job_id = full_qpu_job.job_id()

print("FULL QPU JOB ID:", full_qpu_job_id)

In [ ]:
# ============================================================
# Check full QPU job status without blocking
# ============================================================

full_qpu_job_id = "d834t7ftjchs73bopdsg"

full_qpu_job_reloaded = service.job(full_qpu_job_id)

print("Full QPU job ID:", full_qpu_job_reloaded.job_id())
print("Status:", full_qpu_job_reloaded.status())

try:
    print("Metrics:")
    print(full_qpu_job_reloaded.metrics())
except Exception as e:
    print("Metrics unavailable:", e)

In [ ]:
full_qpu_job_id = "d834t7ftjchs73bopdsg"

full_qpu_job_reloaded = service.job(full_qpu_job_id)

print("Full QPU job ID:", full_qpu_job_reloaded.job_id())
print("Status:", full_qpu_job_reloaded.status())

try:
    print("Metrics:")
    print(full_qpu_job_reloaded.metrics())
except Exception as e:
    print("Metrics unavailable:", e)

In [ ]:
full_qpu_job_id = "d834t7ftjchs73bopdsg"

full_qpu_job_reloaded = service.job(full_qpu_job_id)

print("Full QPU job ID:", full_qpu_job_reloaded.job_id())
print("Status:", full_qpu_job_reloaded.status())

try:
    print("Metrics:")
    print(full_qpu_job_reloaded.metrics())
except Exception as e:
    print("Metrics unavailable:", e)

In [ ]:
# ============================================================
# Retrieve full QPU results
# ============================================================

import numpy as np
import pandas as pd

full_qpu_job_id = "d834t7ftjchs73bopdsg"

full_qpu_job_reloaded = service.job(full_qpu_job_id)

print("Full QPU job ID:", full_qpu_job_reloaded.job_id())
print("Status:", full_qpu_job_reloaded.status())

# Retrieve result now that job is DONE
full_qpu_result = full_qpu_job_reloaded.result()

qpu_rows = []

for i in range(len(full_qpu_result)):
    pub_result = full_qpu_result[i]

    # Extract expectation values
    evs = np.asarray(pub_result.data.evs).ravel()

    print(f"Circuit {i}: evs shape = {evs.shape}")

    assert evs.shape == (180,), f"Expected shape (180,), got {evs.shape}"

    qpu_rows.append(evs)

projections_qpu = np.vstack(qpu_rows)

print("\nprojections_qpu shape:", projections_qpu.shape)

assert projections_qpu.shape == (10, 180)

print("Full QPU result retrieval passed.")

In [ ]:
# ============================================================
# Compare QPU projections against reference projections
# Compute MSE and MAE by measurement basis
# ============================================================

import numpy as np
import pandas as pd

# boundary_train_indices are indices into the original training set.
# So the matching precomputed reference projections come from X_train_quantum.
projections_reference = X_train_quantum.iloc[
    boundary_train_indices, :
].reset_index(drop=True)

projections_reference = projections_reference.to_numpy(dtype=float)

print("projections_reference shape:", projections_reference.shape)
print("projections_qpu shape:", projections_qpu.shape)

assert projections_reference.shape == (10, 180)
assert projections_qpu.shape == (10, 180)

# Basis blocks following our implementation plan:
# X = columns 0-59
# Y = columns 60-119
# Z = columns 120-179
basis_blocks = {
    "X": (0, 60),
    "Y": (60, 120),
    "Z": (120, 180),
}

basis_noise_rows = []

for basis_name, (start, end) in basis_blocks.items():
    ref_block = projections_reference[:, start:end]
    qpu_block = projections_qpu[:, start:end]

    mse = np.mean((qpu_block - ref_block) ** 2)
    mae = np.mean(np.abs(qpu_block - ref_block))
    max_abs_error = np.max(np.abs(qpu_block - ref_block))

    basis_noise_rows.append({
        "Basis": basis_name,
        "MSE": mse,
        "MAE": mae,
        "Max Abs Error": max_abs_error
    })

basis_noise_df = pd.DataFrame(basis_noise_rows).round(6)

basis_noise_df

In [ ]:
# ============================================================
# Combine per-basis F1 scores with QPU-vs-reference noise
# ============================================================

# Extract single-basis F1 scores from phase1_f1_df
basis_f1_df = phase1_f1_df[
    phase1_f1_df["Model"].isin([
        "X-only Quantum",
        "Y-only Quantum",
        "Z-only Quantum"
    ])
].copy()

# Add basis label
basis_f1_df["Basis"] = basis_f1_df["Model"].str[0]

# Merge with noise table
noise_vs_f1_df = basis_f1_df.merge(
    basis_noise_df,
    on="Basis"
)

# Compute F1 drop relative to full quantum model
full_quantum_f1 = phase1_f1_df.loc[
    phase1_f1_df["Model"] == "Full Quantum 180-dim",
    "Weighted F1"
].iloc[0]

noise_vs_f1_df["F1 Drop from Full Quantum"] = (
    full_quantum_f1 - noise_vs_f1_df["Weighted F1"]
)

noise_vs_f1_df = noise_vs_f1_df[
    [
        "Basis",
        "Weighted F1",
        "F1 Drop from Full Quantum",
        "MSE",
        "MAE",
        "Max Abs Error"
    ]
].round(6)

noise_vs_f1_df

In [ ]:
# ============================================================
# Phase 4 Visual: Basis F1 vs QPU-vs-Reference MSE
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

plot_df = noise_vs_f1_df.copy()

basis_labels = plot_df["Basis"].to_numpy()
x = np.arange(len(basis_labels))

fig, ax1 = plt.subplots(figsize=(7, 4))

# Left axis: F1
ax1.bar(x - 0.2, plot_df["Weighted F1"], width=0.4, label="Weighted F1")
ax1.set_ylabel("Weighted F1 Score")
ax1.set_ylim(0.60, 0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(basis_labels)
ax1.set_xlabel("Measurement Basis")

# Right axis: MSE
ax2 = ax1.twinx()
ax2.bar(x + 0.2, plot_df["MSE"], width=0.4, label="QPU-vs-Reference MSE")
ax2.set_ylabel("MSE")
ax2.set_ylim(0.00, max(plot_df["MSE"]) * 1.25)

plt.title("Basis Predictiveness vs Hardware Discrepancy")

# Combine legends
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

plt.show()

In [ ]:
# ============================================================
# Phase 4 Visual 2: Noise heatmap
# Per-basis, per-qubit squared error
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Average squared error across the 10 boundary samples
per_feature_sq_error = np.mean(
    (projections_qpu - projections_reference) ** 2,
    axis=0
)

# Reshape into:
# row 0 = X basis, columns 0-59
# row 1 = Y basis, columns 0-59
# row 2 = Z basis, columns 0-59
noise_heatmap = np.vstack([
    per_feature_sq_error[0:60],
    per_feature_sq_error[60:120],
    per_feature_sq_error[120:180],
])

print("noise_heatmap shape:", noise_heatmap.shape)

plt.figure(figsize=(12, 3))

plt.imshow(noise_heatmap, aspect="auto")
plt.colorbar(label="Mean Squared Error")

plt.yticks([0, 1, 2], ["X", "Y", "Z"])
plt.xlabel("Qubit index")
plt.ylabel("Measurement basis")
plt.title("QPU-vs-Reference Error by Basis and Qubit")

plt.show()

In [ ]:
# ============================================================
# Find highest-error qubits by basis
# ============================================================

top_error_rows = []

basis_names = ["X", "Y", "Z"]

for basis_idx, basis_name in enumerate(basis_names):
    errors = noise_heatmap[basis_idx]

    top_indices = np.argsort(errors)[::-1][:10]

    for rank, qubit_idx in enumerate(top_indices, start=1):
        top_error_rows.append({
            "Basis": basis_name,
            "Rank": rank,
            "Qubit index": qubit_idx,
            "Mean Squared Error": errors[qubit_idx]
        })

top_error_df = pd.DataFrame(top_error_rows)
top_error_df["Mean Squared Error"] = top_error_df["Mean Squared Error"].round(6)

top_error_df

In [ ]:
# ============================================================
# Error distribution summary by basis
# ============================================================

error_distribution_rows = []

for basis_idx, basis_name in enumerate(["X", "Y", "Z"]):
    errors = noise_heatmap[basis_idx]

    error_distribution_rows.append({
        "Basis": basis_name,
        "Mean MSE": np.mean(errors),
        "Median MSE": np.median(errors),
        "Std MSE": np.std(errors),
        "Min MSE": np.min(errors),
        "Max MSE": np.max(errors),
        "Qubits > 0.10 MSE": np.sum(errors > 0.10),
        "Qubits > 0.20 MSE": np.sum(errors > 0.20),
    })

error_distribution_df = pd.DataFrame(error_distribution_rows).round(6)

error_distribution_df

In [ ]:
# ============================================================
# Visual: Error distribution by basis
# ============================================================

import matplotlib.pyplot as plt

x_errors = noise_heatmap[0]
y_errors = noise_heatmap[1]
z_errors = noise_heatmap[2]

plt.figure(figsize=(7, 4))

plt.boxplot(
    [x_errors, y_errors, z_errors],
    labels=["X", "Y", "Z"],
    showmeans=True
)

plt.ylabel("Per-qubit Mean Squared Error")
plt.xlabel("Measurement Basis")
plt.title("Distribution of QPU-vs-Reference Error by Basis")

plt.show()

In [ ]:
# ============================================================
# Final Visual: Noise vs Predictive Performance by Basis
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

plot_df = noise_vs_f1_df.copy()

x = plot_df["MSE"].to_numpy()
y = plot_df["Weighted F1"].to_numpy()
labels = plot_df["Basis"].to_numpy()

plt.figure(figsize=(6, 4))

plt.scatter(x, y, s=100)

for i, label in enumerate(labels):
    plt.annotate(
        label,
        (x[i], y[i]),
        textcoords="offset points",
        xytext=(7, 7),
        fontsize=12
    )

plt.xlabel("QPU-vs-Reference MSE")
plt.ylabel("Weighted F1 Score")
plt.title("Predictive Performance vs Hardware Discrepancy")

# Optional correlation, but be careful because n=3
corr = np.corrcoef(x, y)[0, 1]
plt.text(
    0.05,
    0.95,
    f"Correlation = {corr:.3f}\n(n = 3 bases)",
    transform=plt.gca().transAxes,
    va="top"
)

plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

plt.scatter(x, y, s=100)

for i, label in enumerate(labels):
    plt.annotate(
        label,
        (x[i], y[i]),
        textcoords="offset points",
        xytext=(7, 7),
        fontsize=12
    )

plt.xlabel("QPU-vs-Reference MSE")
plt.ylabel("Weighted F1 Score")
plt.title("Basis Signal Strength vs QPU Discrepancy")

plt.text(
    0.05,
    0.95,
    "Descriptive only: n = 3 bases",
    transform=plt.gca().transAxes,
    va="top"
)

plt.show()

## Final Finding: Hardware Noise Resilience Across Measurement Bases

We tested whether the predictive usefulness of quantum-projected features depends on measurement basis. The full 180-dimensional quantum model achieved the highest weighted F1 score of 0.8108. Among individual measurement bases, the Z basis was strongest with F1 = 0.7833, compared with X = 0.6889 and Y = 0.7029.

We then selected 10 boundary samples from the classical SVM support vectors and ran their 60-qubit ZZFeatureMap circuits on IBM quantum hardware. We compared the QPU outputs against the precomputed reference projections for the same samples.

The QPU-vs-reference MSE was:

- X basis: 0.0491
- Y basis: 0.0494
- Z basis: 0.1493

This means the Z basis was both the most predictive and the most hardware-discrepant. Therefore, the simplest hypothesis — that the best basis is simply the least noisy basis — was not supported. Instead, the results suggest that the Z basis carries stronger label-relevant signal, enough that it remains the most useful single basis despite higher QPU discrepancy.

A limitation is that the reference projections are precomputed quantum projections, not a true noiseless 60-qubit statevector simulation. Therefore, this experiment measures QPU-vs-reference discrepancy rather than pure ideal-vs-noisy hardware error.

In [ ]:
# ============================================================
# Polishing Chunk 1: Save final results and QPU outputs
# ============================================================

import os
import numpy as np
import pandas as pd

# Create folders
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# -----------------------------
# Save main result tables
# -----------------------------

phase1_f1_df.to_csv("results/phase1_f1_scores.csv", index=False)
basis_noise_df.to_csv("results/basis_noise_table.csv", index=False)
noise_vs_f1_df.to_csv("results/noise_vs_f1_table.csv", index=False)
error_distribution_df.to_csv("results/error_distribution_by_basis.csv", index=False)
top_error_df.to_csv("results/top_error_qubits_by_basis.csv", index=False)

# -----------------------------
# Save QPU and reference projections
# -----------------------------

pd.DataFrame(projections_qpu).to_csv(
    "results/projections_qpu_boundary_samples.csv",
    index=False
)

pd.DataFrame(projections_reference).to_csv(
    "results/projections_reference_boundary_samples.csv",
    index=False
)

# -----------------------------
# Save compact npz archive
# -----------------------------

np.savez(
    "results/qpu_experiment_outputs.npz",
    projections_qpu=projections_qpu,
    projections_reference=projections_reference,
    boundary_train_indices=boundary_train_indices,
    boundary_labels=boundary_labels,
    full_qpu_job_id=full_qpu_job_id
)

print("Saved result tables:")
print("- results/phase1_f1_scores.csv")
print("- results/basis_noise_table.csv")
print("- results/noise_vs_f1_table.csv")
print("- results/error_distribution_by_basis.csv")
print("- results/top_error_qubits_by_basis.csv")

print("\nSaved QPU outputs:")
print("- results/projections_qpu_boundary_samples.csv")
print("- results/projections_reference_boundary_samples.csv")
print("- results/qpu_experiment_outputs.npz")